# Step 10: export the selected model

Run after notebook 09 writes `results/final_selection.json`. Package the actual selected ranker, feature order, candidate settings and optional HMM configuration. Do not fit anything here.

The bundle is prepared for API integration. It does not start or deploy a server. Teammates' stacking artifacts must be supplied separately before a stacking model can be exported.

In [1]:
from pathlib import Path
import json
import shutil
import hashlib
import zipfile
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
import nbformat
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb

selection_path = Path("results/final_selection.json")
assert selection_path.exists(), "Run notebook 09 before exporting."
selection = json.loads(selection_path.read_text())
assert "stack" not in selection["selected_name"].lower(), "Stacking requires all four base models, combiner and real preprocessing artifacts."
model_path = Path(selection["model_path"]) if "model_path" in selection else Path(selection["model_dir"]) / "model.json"
assert model_path.is_file()
contract = json.loads(Path("outputs/folds/contract.json").read_text())
feature_columns = contract["feature_columns"]
hmm_enabled = bool(selection["hmm_enabled"])
bundle = Path("exports/final_model")
bundle.mkdir(parents=True, exist_ok=True)

## Copy the model and portable settings

Candidate and feature code never needs reference routes or labels. The training flag threshold is evaluation metadata, not an inference filter: every incoming GPS point stays available to candidate generation and route decoding.

Keep predictor columns in the exported order, use NaN for missing values, and add the geometric distance ratio before prediction. Softmax is applied separately inside each GPS-point group.

In [2]:
shutil.copy2(model_path, bundle / "ranker.json")
(bundle / "feature_columns.json").write_text(json.dumps(feature_columns, indent=2) + "\n")
settings = contract["candidate_settings"]
candidate_settings = {"radius_m": settings["radius_m"], "K": settings["K"],
    "ordering": ["group_id", "distance", "segment_id"], "segment_id": "zero-based row in the trip's arcs file",
    "empty_shortlist": "retain point in point metadata; decoder handles missing observations explicitly"}
(bundle / "candidate_settings.json").write_text(json.dumps(candidate_settings, indent=2) + "\n")
preprocessing = {"coordinate_order": ["longitude", "latitude"],
    "projection": "WGS84 to per-trip UTM; zone=int((mean_lon+180)//6)+1; EPSG=(32600 north or 32700 south)+zone; always_xy=True",
    "directed_arcs": True, "distance_units": "metres", "feature_dtype": "float32", "missing_values": "NaN; no imputer or scaler for ranker",
    "heading": "grid-north angle from previous GPS point to next GPS point; 0..360 degrees",
    "stationary_cutoff_mps": 0.5, "heading_neighbour_min_displacement_m": 1.0,
    "context_offsets": [-2, -1, 1, 2], "matching_mode": "whole-trip; future observations required",
    "distance_ratio": contract["distance_ratio"], "softmax_temperature": 1.0,
    "softmax": "per GPS group: exp(raw_score-max(raw_score))/sum(exp(raw_score-max(raw_score)))",
    "inference_inputs": ["nodes(lon,lat)", "directed arcs(start,end)", "track(lon,lat,time)"],
    "reference_route_required": False, "bad_gps_reference_filter_at_inference": False}
(bundle / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2) + "\n")
if hmm_enabled:
    hmm_path = Path(selection["hmm_config_path"])
    assert hmm_path.is_file()
    hmm_config = json.loads(hmm_path.read_text())
    hmm_config["enabled"] = True
else:
    hmm_config = {"enabled": False, "reason": "Final evaluation selected the classifier-only route."}
(bundle / "hmm_config.json").write_text(json.dumps(hmm_config, indent=2) + "\n")

91

## Include the implementation and versions

The feature helper contains definitions only and no training outputs. Include the shared HMM implementation only when it is enabled. Clear notebook outputs so no training data is embedded in the bundle.

In [3]:
helpers = ["05_feature_helpers.ipynb"]
if hmm_enabled:
    helpers.append("08_hmm_helpers.ipynb")
for filename in helpers:
    notebook = nbformat.read(filename, as_version=4)
    for cell in notebook.cells:
        if cell.cell_type == "code":
            cell.outputs = []
            cell.execution_count = None
    nbformat.write(notebook, bundle / filename)
packages = ["numpy", "pandas", "shapely", "pyproj", "pyarrow", "xgboost", "scipy", "nbformat"]
(bundle / "requirements.txt").write_text("\n".join(f"{name}=={version(name)}" for name in packages) + "\n")
readme = """# PathSnap exported ranker

Load ranker.json with XGBRanker.load_model. Select predictor columns in the exact order in feature_columns.json.
Generate candidates with candidate_settings.json and features with 05_feature_helpers.ipynb; add distance_ratio from preprocessing.json.
Apply stable softmax separately per GPS group. Retain empty and singleton observations in the trip metadata.
If hmm_config.json has enabled=true, run the included 08_hmm_helpers.ipynb decoder with those frozen settings.
The feature context uses future points, so requests must provide a whole trip.

Only nodes, directed arcs and GPS observations are prediction inputs. No reference route or point labels are required.
This is an integration bundle, not a running API. Review manifest.json for validation, selection and timing scope.
"""
(bundle / "README.md").write_text(readme)

812

## Reload and verify the exported model

Use a small feature-only sample to check model format, feature order, finite predictions and per-point normalization. Reference labels are not loaded. Compare exported predictions with the source model to detect copy or version problems.

In [4]:
base_columns = [name for name in feature_columns if name != "distance_ratio"]
sample = next(pq.ParquetFile("outputs/features/features.parquet").iter_batches(batch_size=256,
    columns=["group_id"] + base_columns)).to_pandas()
nearest = sample["distance"] - sample["distance_above_nearest_m"]
sample["distance_ratio"] = ((sample["distance"] + 1) / (nearest + 1)).astype("float32")
model = xgb.XGBRanker(n_jobs=2)
model.load_model(bundle / "ranker.json")
assert model.get_booster().feature_names == feature_columns
source_model = xgb.XGBRanker(n_jobs=2)
source_model.load_model(model_path)
raw = model.predict(sample[feature_columns])
np.testing.assert_allclose(raw, source_model.predict(sample[feature_columns]), atol=0, rtol=0)
assert np.isfinite(raw).all()
values = pd.Series(raw.astype(float))
groups = sample["group_id"].to_numpy()
weights = np.exp(values - values.groupby(groups).transform("max"))
probability = weights / weights.groupby(groups).transform("sum")
np.testing.assert_allclose(probability.groupby(groups).sum(), 1, atol=1e-12)
validation = {"model_reloaded": True, "feature_order_matches": True, "source_predictions_equal": True,
              "sample_rows": len(sample), "finite_scores": True, "group_probability_sums": True}

## Checksums and archive

Manifest artifact paths are relative to the bundle root. Verify every checksum before producing the zip. The final selection may use student-test evidence for model choice; subsequent evaluation is not a fresh untouched test.

In [5]:
files = ["ranker.json", "feature_columns.json", "candidate_settings.json", "preprocessing.json",
         "hmm_config.json", "requirements.txt", "README.md"] + helpers
checksums = {name: hashlib.sha256((bundle / name).read_bytes()).hexdigest() for name in files}
manifest = {"created_at": datetime.now(ZoneInfo("Asia/Kolkata")).isoformat(),
    "selected_model": selection["selected_name"], "hmm_enabled": hmm_enabled,
    "selection_rationale": selection["rationale"], "selection_metrics": selection.get("metrics", {}),
    "timing_scope": selection.get("timing_scope", "See final evaluation; inference timings use precomputed features."),
    "feature_count": len(feature_columns), "artifacts": checksums, "validation": validation,
    "deployment_status": "artifact bundle prepared; API not deployed",
    "stacking_status": "pending teammates' three base-model artifacts and combiner"}
(bundle / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
for filename, expected in manifest["artifacts"].items():
    assert not Path(filename).is_absolute()
    assert hashlib.sha256((bundle / filename).read_bytes()).hexdigest() == expected
archive = bundle.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
    for filename in files + ["manifest.json"]:
        zipped.write(bundle / filename, arcname=filename)
print("Exported:", bundle, "| Archive:", archive)
print("Validated:", validation)
print("Selected:", selection["selected_name"], "| HMM enabled:", hmm_enabled)

Exported: exports/final_model | Archive: exports/final_model.zip
Validated: {'model_reloaded': True, 'feature_order_matches': True, 'source_predictions_equal': True, 'sample_rows': 256, 'finite_scores': True, 'group_probability_sums': True}
Selected: ranker | HMM enabled: False
